# NB-06: DiTBlock, Patchify, and the Head Module

## Learning Objectives
- Trace DiTBlock.forward line-by-line: how five primitives (norm, self-attention, cross-attention, adaLN-Zero, FFN) wire together with residual connections
- Understand patchify: how Conv3d(16->384, kernel=(1,2,2), stride=(1,2,2)) converts video latents into token sequences
- See how Head applies 2-parameter adaLN to project from model dim back to output channels, then unpatchify reconstructs the video shape

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm, modulate function), NB-04 (SelfAttention, CrossAttention), NB-05 (adaLN-Zero, GateModule)
- **Assumed concepts:** Residual connections, adaLN-Zero gate=0 identity, Conv3d kernel/stride mechanics

## Concept Map
- **DiTBlock** assembles Phase 1 primitives into the core repeated unit, stacked N times in WanModel (NB-07)
- **Patchify** converts video latents to token sequences for DiT block processing (NB-07)
- **Head** + **unpatchify** convert DiT block output back to video shape (NB-07)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# NB-06 imports
from diffsynth.models.wan_video_dit import (
    DiTBlock, GateModule, Head, modulate, precompute_freqs_cis_3d
)
import torch
import torch.nn as nn
import math
from einops import rearrange

print("Setup complete.")

## DiTBlock Data Flow (5 primitives + residual connections)

```
     x [B, S, dim]     t_mod [B, 6, dim]     freqs [S, 1, head_dim//2]
          |                    |
          |     modulation + t_mod -> chunk(6) -> six [B, 1, dim] params
          |          |         |         |           |         |
          |     shift_msa  scale_msa  gate_msa  shift_mlp  scale_mlp  gate_mlp
          |          |         |         |
          +---> norm1 ---> modulate(shift_msa, scale_msa)
          |         |
          |    self_attn(input_x, freqs)    # SelfAttention with RoPE (NB-04)
          |         |
          |    gate(x, gate_msa, sa_out)    # GateModule: x + gate * residual (NB-05)
          |         |
          +-------> x'
          |
          +---> norm3(x') ---> cross_attn(., context)    # CrossAttention (NB-04)
          |         |
          |    x' + ca_out                  # Simple residual -- NO gate, NO adaLN
          |         |
          +-------> x''
          |                    |         |
          +---> norm2 ---> modulate(shift_mlp, scale_mlp)
          |         |
          |    ffn(input_x)                 # Linear -> GELU -> Linear
          |         |
          |    gate(x'', gate_mlp, ffn_out) # GateModule: x'' + gate * residual
          |         |
          +-------> output [B, S, dim]
```

**Key: self-attention and FFN use adaLN + gated residual. Cross-attention uses SIMPLE residual -- no adaLN, no gate.**

Notice the asymmetry: six modulation parameters split into two groups of three. Self-attention and FFN each consume (shift, scale, gate). Cross-attention consumes **none** -- it gets only a `norm3` (which has its own learnable weight and bias, unlike norm1/norm2).

Source: `diffsynth/models/wan_video_dit.py`, lines 215-231

## 1. DiTBlock Sub-Module Inventory

The DiTBlock is the **core repeated unit** of the diffusion backbone. WanModel stacks N of these blocks (30 in the production 1.3B model, 2 in our reduced config). Each block wires together all five primitives from Phase 1: normalization, self-attention, cross-attention, adaLN-Zero modulation, and feed-forward network.

**Sub-modules:**

| Sub-module | Type | Purpose |
|------------|------|---------|
| `self_attn` | SelfAttention | Token-to-token attention with 3D RoPE (NB-04) |
| `cross_attn` | CrossAttention | Token-to-text attention for text conditioning (NB-04) |
| `norm1` | LayerNorm(**affine=False**) | Normalize before self-attention -- **0 learnable parameters** |
| `norm2` | LayerNorm(**affine=False**) | Normalize before FFN -- **0 learnable parameters** |
| `norm3` | LayerNorm(**affine=True**) | Normalize before cross-attention -- **768 learnable parameters** (weight + bias for dim=384) |
| `ffn` | Sequential(Linear, GELU, Linear) | Feed-forward network: dim -> ffn_dim -> dim |
| `modulation` | Parameter [1, 6, dim] | Per-block learned offset for six adaLN parameters (NB-05) |
| `gate` | GateModule | Gated residual: `x + gate * residual` (NB-05) |

**Norm asymmetry (Pitfall 3):** `norm1` and `norm2` have `elementwise_affine=False` because the `modulate` function provides their scale and shift via the adaLN mechanism. `norm3` has `elementwise_affine=True` (default) because the cross-attention path has **no modulation** -- it needs its own learnable weight and bias to compensate.

Source: `diffsynth/models/wan_video_dit.py`, lines 197-213

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 197-213
dim, num_heads, ffn_dim = 384, 4, 1024  # reduced config

block = DiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)

print("DiTBlock sub-modules and parameter counts:")
print(f"  self_attn.q  (Linear {dim}->{dim}):  {sum(p.numel() for p in block.self_attn.q.parameters()):,} params")
print(f"  self_attn.k  (Linear {dim}->{dim}):  {sum(p.numel() for p in block.self_attn.k.parameters()):,} params")
print(f"  self_attn.norm_q (RMSNorm {dim}):    {sum(p.numel() for p in block.self_attn.norm_q.parameters()):,} params")
print(f"  self_attn.norm_k (RMSNorm {dim}):    {sum(p.numel() for p in block.self_attn.norm_k.parameters()):,} params")
print(f"  self_attn.v  (Linear {dim}->{dim}):  {sum(p.numel() for p in block.self_attn.v.parameters()):,} params")
print(f"  self_attn.o  (Linear {dim}->{dim}):  {sum(p.numel() for p in block.self_attn.o.parameters()):,} params")
print(f"  cross_attn.q (Linear {dim}->{dim}):  {sum(p.numel() for p in block.cross_attn.q.parameters()):,} params")
print(f"  cross_attn.k (Linear {dim}->{dim}):  {sum(p.numel() for p in block.cross_attn.k.parameters()):,} params")
print(f"  cross_attn.v (Linear {dim}->{dim}):  {sum(p.numel() for p in block.cross_attn.v.parameters()):,} params")
print(f"  cross_attn.o (Linear {dim}->{dim}):  {sum(p.numel() for p in block.cross_attn.o.parameters()):,} params")
print(f"  norm1 (elementwise_affine=False):     {sum(p.numel() for p in block.norm1.parameters()):,} params")
print(f"  norm2 (elementwise_affine=False):     {sum(p.numel() for p in block.norm2.parameters()):,} params")
print(f"  norm3 (elementwise_affine=True):      {sum(p.numel() for p in block.norm3.parameters()):,} params")
print(f"  ffn[0] (Linear {dim}->{ffn_dim}):     {sum(p.numel() for p in block.ffn[0].parameters()):,} params")
print(f"  ffn[2] (Linear {ffn_dim}->{dim}):     {sum(p.numel() for p in block.ffn[2].parameters()):,} params")
print(f"  modulation [1, 6, {dim}]:             {block.modulation.numel():,} params")
print(f"  gate (GateModule):                    {sum(p.numel() for p in block.gate.parameters()):,} params")
total = sum(p.numel() for p in block.parameters())
print(f"  -----------------------------------------")
print(f"  Block total: {total:,}")

# Verify key asymmetries
assert sum(p.numel() for p in block.norm1.parameters()) == 0, "norm1 should have 0 params"
assert sum(p.numel() for p in block.norm2.parameters()) == 0, "norm2 should have 0 params"
assert sum(p.numel() for p in block.norm3.parameters()) == 2 * dim, f"norm3 should have {2*dim} params"
print(f"\nnorm1 params: 0 (elementwise_affine=False -- modulate provides scale/shift)")
print(f"norm3 params: {2*dim} (elementwise_affine=True -- cross-attn has no modulation)")

## 2. DiTBlock Forward Pass

The forward pass has four stages, matching the ASCII diagram above:

1. **Stage 1 -- Six modulation parameters:** The per-block `modulation` offset is added to `t_mod` (the time-embedding projection from WanModel), then chunked into six [B, 1, dim] parameters: shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp.

2. **Stage 2 -- Self-attention with adaLN + gated residual:** Apply `modulate(norm1(x), shift_msa, scale_msa)` to condition the input, run `self_attn` with 3D RoPE frequencies, then `gate(x, gate_msa, sa_out)` to apply gated residual.

3. **Stage 3 -- Cross-attention with simple residual:** Apply `norm3(x)` (note: norm3 has learnable params), run `cross_attn` with the text context, then add the result directly: `x = x + cross_attn_out`. **No gate, no adaLN modulation.**

4. **Stage 4 -- FFN with adaLN + gated residual:** Apply `modulate(norm2(x), shift_mlp, scale_mlp)`, run `ffn`, then `gate(x, gate_mlp, ffn_out)`.

Note: Line 216 has a `has_seq = len(t_mod.shape) == 4` branch for S2V's per-position timestep modulation (Phase 5). The standard WanModel path always uses `has_seq=False` (t_mod shape is [B, 6, dim], 3D).

Source: `diffsynth/models/wan_video_dit.py`, lines 215-231

### Frequency Tensor Construction for Standalone DiTBlock Testing

To test a DiTBlock in isolation, we need to construct the 3D RoPE frequency tensor that is normally assembled by `WanModel.forward` (covered in NB-07). The frequency tensor encodes each token's 3D position (frame, height, width) in the video grid.

The assembly follows NB-03's frequency precomputation:
1. `precompute_freqs_cis_3d(head_dim)` returns three frequency tensors (f_freqs, h_freqs, w_freqs), each for one spatial axis
2. We slice each to the grid dimensions (F, H, W), expand, and concatenate along the last dimension
3. The result is reshaped to `[seq_len, 1, head_dim//2]` for broadcasting across attention heads

For our reduced config (patchified grid F=4, H=4, W=4): `seq_len = 4*4*4 = 64`.

Source: `diffsynth/models/wan_video_dit.py`, lines 377-381 (WanModel.forward)

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 215 (DiTBlock.forward)
# Freqs assembly from lines 377-381 (WanModel.forward)
dim, num_heads, ffn_dim = 384, 4, 1024
B, S = 1, 64  # seq_len = 4*4*4 (patchified grid)
head_dim = dim // num_heads  # 96

block = DiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)
block.eval()

# Inputs
x       = torch.randn(B, S, dim)       # [B, S, dim] -- video tokens
context = torch.randn(B, 10, dim)      # [B, S_text, dim] -- text embedding output
t_mod   = torch.randn(B, 6, dim)       # [B, 6, dim] -- time_projection output

# Build 3D RoPE freqs for seq_len=64 (grid 4x4x4)
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
F, H, W = 4, 4, 4
freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),
], dim=-1).reshape(F * H * W, 1, -1)
assert freqs.shape == torch.Size([F * H * W, 1, head_dim // 2]), (
    f"Expected freqs [{F*H*W}, 1, {head_dim//2}], got {freqs.shape}")
print(f"freqs shape: {freqs.shape}  -- [seq_len, 1, head_dim//2]")

# Single forward pass
with torch.no_grad():
    out = block(x, context, t_mod, freqs)
assert out.shape == torch.Size([B, S, dim]), f"Expected {(B, S, dim)}, got {out.shape}"
print(f"Input:  x {x.shape}, context {context.shape}, t_mod {t_mod.shape}")
print(f"Output: {out.shape}  (shape preserved through DiTBlock)")

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 219-231
# Annotated line-by-line forward trace with shape prints at each stage

# Re-use block, x, context, t_mod, freqs from previous cell

# ---- Stage 1: Six modulation parameters ----
# modulation [1, 6, dim] + t_mod [B, 6, dim] -> chunk(6) -> six [B, 1, dim]
shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = (
    block.modulation.to(dtype=t_mod.dtype, device=t_mod.device) + t_mod
).chunk(6, dim=1)
print(f"Stage 1 -- Six modulation parameters:")
print(f"  shift_msa: {shift_msa.shape}")
print(f"  scale_msa: {scale_msa.shape}")
print(f"  gate_msa:  {gate_msa.shape}")
print(f"  shift_mlp: {shift_mlp.shape}")
print(f"  scale_mlp: {scale_mlp.shape}")
print(f"  gate_mlp:  {gate_mlp.shape}")
assert shift_msa.shape == torch.Size([B, 1, dim])

# ---- Stage 2: Self-attention with adaLN + gated residual ----
# norm1(x) has elementwise_affine=False -> no learnable params
# modulate applies shift and scale from adaLN
input_x_sa = modulate(block.norm1(x), shift_msa, scale_msa)
print(f"\nStage 2 -- Self-attention with adaLN + gated residual:")
print(f"  norm1(x):      {block.norm1(x).shape}")
print(f"  modulate():    {input_x_sa.shape}")
with torch.no_grad():
    sa_out = block.self_attn(input_x_sa, freqs)
    x_after_sa = block.gate(x, gate_msa, sa_out)
print(f"  self_attn():   {sa_out.shape}")
print(f"  gate():        {x_after_sa.shape}  (x + gate_msa * sa_out)")
assert x_after_sa.shape == torch.Size([B, S, dim])

# ---- Stage 3: Cross-attention with simple residual (no gate, no adaLN) ----
# norm3 has elementwise_affine=True (768 learnable params for dim=384)
# Cross-attention uses SIMPLE additive residual: x = x + cross_attn(norm3(x), context)
with torch.no_grad():
    ca_out = block.cross_attn(block.norm3(x_after_sa), context)
    x_after_ca = x_after_sa + ca_out  # simple residual -- no gate, no adaLN
print(f"\nStage 3 -- Cross-attention with simple residual (no gate, no adaLN):")
print(f"  norm3(x):      {block.norm3(x_after_sa).shape}  (norm3 has {sum(p.numel() for p in block.norm3.parameters())} learnable params)")
print(f"  cross_attn():  {ca_out.shape}")
print(f"  x + ca_out:    {x_after_ca.shape}  (simple residual -- no gate)")
assert x_after_ca.shape == torch.Size([B, S, dim])

# ---- Stage 4: FFN with adaLN + gated residual ----
input_x_ffn = modulate(block.norm2(x_after_ca), shift_mlp, scale_mlp)
print(f"\nStage 4 -- FFN with adaLN + gated residual:")
print(f"  norm2(x):      {block.norm2(x_after_ca).shape}")
print(f"  modulate():    {input_x_ffn.shape}")
with torch.no_grad():
    ffn_out = block.ffn(input_x_ffn)
    x_final = block.gate(x_after_ca, gate_mlp, ffn_out)
print(f"  ffn():         {ffn_out.shape}")
print(f"  gate():        {x_final.shape}  (x'' + gate_mlp * ffn_out)")
assert x_final.shape == torch.Size([B, S, dim]), f"Expected {(B, S, dim)}, got {x_final.shape}"
print(f"\nFinal output: {x_final.shape} == input: {x.shape} -- shape preserved")

## 3. Patchify: Video Latents to Token Sequence

Before video latents can be processed by DiTBlocks, they must be converted from a 5D volume `[B, C, F, H, W]` into a 2D token sequence `[B, S, dim]`. This is the job of **patchify**: a `Conv3d` followed by a flatten.

**Shape math:**

```
Conv3d(in_channels=16, out_channels=384, kernel=(1,2,2), stride=(1,2,2))

Input:  [B, 16, F, H, W]      e.g. [1, 16, 4, 8, 8]
Conv3d: [B, 384, F, H/2, W/2] e.g. [1, 384, 4, 4, 4]
Flatten: [B, F*H/2*W/2, 384]  e.g. [1, 64, 384]
```

**Why temporal kernel=1:** The VAE already compresses time by 4x (e.g., 21 video frames become ~5 latent frames). Patchify only compresses the spatial dimensions with a 2x2 kernel, preserving temporal resolution.

**Sequence length formula:** `seq_len = F * (H/2) * (W/2)`. For our reduced input `(F=4, H=8, W=8)`: `seq_len = 4 * 4 * 4 = 64`.

**Impact on attention cost:** Self-attention is O(S^2) in sequence length. Spatial 2x2 patching reduces S by 4x (from `F*H*W` to `F*(H/2)*(W/2)`), reducing attention cost by approximately 16x. For production resolution (21 frames, 480p latents): without patching the sequence would be ~1.6M tokens; with (1,2,2) patching it drops to ~403K tokens.

Source: `diffsynth/models/wan_video_dit.py`, lines 307, 340

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 307, 340
patch_size = (1, 2, 2)
in_dim, dim = 16, 384  # reduced config: in_dim=16 (noise only), NOT 48

patch_embedding = nn.Conv3d(in_dim, dim, kernel_size=patch_size, stride=patch_size)

B, C, F, H, W = 1, 16, 4, 8, 8
x = torch.randn(B, C, F, H, W)

with torch.no_grad():
    px = patch_embedding(x)  # [1, 384, 4, 4, 4]
assert px.shape == torch.Size([B, dim, F, H // 2, W // 2]), (
    f"Expected [{B}, {dim}, {F}, {H//2}, {W//2}], got {px.shape}")
print(f"Conv3d output: {px.shape}  -- [B, dim, F, H/2, W/2]")

b, c, f, h, w = px.shape
x_seq = rearrange(px, 'b c f h w -> b (f h w) c')  # [1, 64, 384]
assert x_seq.shape == torch.Size([B, f * h * w, dim]), (
    f"Expected [{B}, {f*h*w}, {dim}], got {x_seq.shape}")
print(f"Flattened:     {x_seq.shape}  -- [B, seq_len, dim]")
print(f"seq_len = {f} * {h} * {w} = {f*h*w}")

# Parameter count: weight [384, 16, 1, 2, 2] + bias [384]
param_count = sum(p.numel() for p in patch_embedding.parameters())
expected = dim * in_dim * math.prod(patch_size) + dim  # weight + bias
assert param_count == expected, f"Expected {expected}, got {param_count}"
print(f"\npatch_embedding params: {param_count:,} = {dim}*{in_dim}*{math.prod(patch_size)} + {dim}")

### ASCII Patch Tiling Grid

Visualizing how a single spatial slice (one frame) is tiled by the (1,2,2) patching kernel:

```
Input spatial slice (H=8, W=8):           After 2x2 patching (4x4 patches):
+--+--+--+--+--+--+--+--+                +-----+-----+-----+-----+
|  |  |  |  |  |  |  |  |                | p00 | p01 | p02 | p03 |
+--+--+--+--+--+--+--+--+    Conv3d      +-----+-----+-----+-----+
|  |  |  |  |  |  |  |  |   -------->    | p10 | p11 | p12 | p13 |
+--+--+--+--+--+--+--+--+   (1,2,2)      +-----+-----+-----+-----+
|  |  |  |  |  |  |  |  |                | p20 | p21 | p22 | p23 |
+--+--+--+--+--+--+--+--+                +-----+-----+-----+-----+
|  |  |  |  |  |  |  |  |                | p30 | p31 | p32 | p33 |
+--+--+--+--+--+--+--+--+                +-----+-----+-----+-----+
|  |  |  |  |  |  |  |  |                4 rows x 4 cols = 16 patches per frame
+--+--+--+--+--+--+--+--+                x 4 frames = 64 tokens total
|  |  |  |  |  |  |  |  |
+--+--+--+--+--+--+--+--+
|  |  |  |  |  |  |  |  |
+--+--+--+--+--+--+--+--+
|  |  |  |  |  |  |  |  |
+--+--+--+--+--+--+--+--+
8 rows x 8 cols = 64 pixels per frame
```

Each 2x2 spatial region (across 16 input channels) is projected to a single 384-dimensional token by the Conv3d kernel. The temporal kernel is 1, so each frame is processed independently -- the 4 frames produce 4 independent spatial grids of tokens.

### ViT 2D vs WAN 3D Patching Comparison

| Aspect | ViT (2D images) | WAN (3D video) |
|--------|----------------|----------------|
| Patch kernel | `(P, P)` e.g. (16, 16) | `(1, 2, 2)` |
| Input dims | `[B, C, H, W]` | `[B, C, F, H, W]` |
| Temporal handling | N/A (single image) | kernel=1 preserves temporal resolution |
| Typical compression | 16x16 = 256x spatial | 2x2 = 4x spatial |
| Sequence length | `(H/P) * (W/P)` | `F * (H/2) * (W/2)` |

WAN uses much smaller spatial patches (2x2 vs 16x16) because the VAE has already compressed the spatial dimensions significantly. The ViT operates on raw pixels, so it needs larger patches to keep sequence length manageable.

### Attention Cost

Sequence length drops from `F*H*W` to `F*(H/2)*(W/2)`, a 4x reduction. For production resolution (21 frames at 480p latent size ~60x90):
- Without patching: `21 * 60 * 90 = 113,400` tokens
- With (1,2,2) patching: `21 * 30 * 45 = 28,350` tokens
- Attention cost reduction: ~16x (since attention is O(S^2))

## 4. Unpatchify: Token Sequence Back to Video Shape

After DiT blocks process the token sequence, the result must be converted back to video shape. The **unpatchify** operation reverses patchify using an einops `rearrange`:

```python
rearrange(x, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
          f=grid_f, h=grid_h, w=grid_w, x=1, y=2, z=2)
```

This reverses both the flatten and the Conv3d compression:
1. The `(f h w)` decomposition splits the sequence dimension back into the 3D grid
2. The `(x y z c)` decomposition splits each token's features back into spatial sub-patches and output channels
3. The `(f x)`, `(h y)`, `(w z)` products reconstruct the original spatial dimensions

Source: `diffsynth/models/wan_video_dit.py`, line 348

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 348
# Unpatchify demo + patchify round-trip verification

out_dim = 16
patch_size = (1, 2, 2)

# Simulate Head output: each token has out_dim * prod(patch_size) features
head_output = torch.randn(1, 64, out_dim * math.prod(patch_size))  # [1, 64, 64]
assert head_output.shape == torch.Size([1, 64, 64])
print(f"Head output: {head_output.shape}  -- [B, seq_len, out_dim*prod(patch_size)]")
print(f"  out_dim * prod(patch_size) = {out_dim} * {math.prod(patch_size)} = {out_dim * math.prod(patch_size)}")

# Unpatchify: rearrange back to video shape
x_video = rearrange(
    head_output, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
    f=4, h=4, w=4, x=patch_size[0], y=patch_size[1], z=patch_size[2]
)
assert x_video.shape == torch.Size([1, 16, 4, 8, 8]), (
    f"Expected [1, 16, 4, 8, 8], got {x_video.shape}")
print(f"Unpatchified: {x_video.shape}  -- [B, out_dim, F, H, W]")

# Full round-trip verification: patchify -> (skip DiT) -> Head -> unpatchify
print(f"\n--- Full Round-Trip Verification ---")
original_input = torch.randn(1, 16, 4, 8, 8)
print(f"1. Original input:  {original_input.shape}")

# Patchify
with torch.no_grad():
    conv_out = patch_embedding(original_input)  # reuse patch_embedding from previous cell
b, c, f, h, w = conv_out.shape
tokens = rearrange(conv_out, 'b c f h w -> b (f h w) c')
print(f"2. After patchify:  {tokens.shape}")

# Simulate Head: Linear(384 -> 64)
head_linear = nn.Linear(dim, out_dim * math.prod(patch_size))
with torch.no_grad():
    head_out = head_linear(tokens)
print(f"3. After Head:      {head_out.shape}")

# Unpatchify
reconstructed = rearrange(
    head_out, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
    f=f, h=h, w=w, x=patch_size[0], y=patch_size[1], z=patch_size[2]
)
print(f"4. After unpatchify: {reconstructed.shape}")
assert reconstructed.shape == original_input.shape, (
    f"Round-trip failed: {reconstructed.shape} != {original_input.shape}")
print(f"\nRound-trip shape preserved: {reconstructed.shape} == {original_input.shape}")

## 5. Head Module: From Model Dim Back to Output

The **Head** module is the final stage of the diffusion backbone. It projects each token from model dimension (`dim=384`) to the output dimension needed for unpatchify (`out_dim * prod(patch_size) = 16 * 4 = 64`).

**Head applies 2-parameter modulation (shift, scale) -- NOT 6 parameters like DiTBlock.** This is a critical distinction:

| Component | Modulation params | Input signal | Shape |
|-----------|------------------|--------------|-------|
| DiTBlock | 6 (shift, scale, gate) x 2 branches | `t_mod` from `time_projection` | `[B, 6, dim]` |
| Head | 2 (shift, scale) | `t` from `time_embedding` | `[B, dim]` |

**Head receives `t` [B, dim] from `time_embedding`, NOT `t_mod` [B, 6, dim] from `time_projection`.** This is confirmed at line 406: `self.head(x, t)`. The variable name `t_mod` in Head.forward's parameter list is misleading -- it actually receives `t`.

**Pipeline:**
1. `norm(x)` -- LayerNorm with `elementwise_affine=False`
2. `modulate(norm_x, shift, scale)` -- 2-parameter adaLN conditioning
3. `Linear(dim, out_dim * prod(patch_size))` -- project to output features

The output dim calculation: `out_dim * prod(patch_size) = 16 * (1*2*2) = 64`. Each token's 384-dimensional representation is projected to 64 dimensions, which unpatchify then rearranges back to `[B, 16, F, H, W]`.

> The VACE encoder reuses DiTBlock but has its own Head-like output projection -- covered in Track 4.

Source: `diffsynth/models/wan_video_dit.py`, lines 254-270, line 406

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 254-270, line 406
# Head module demo using t (NOT t_mod)

head = Head(dim=384, out_dim=16, patch_size=(1, 2, 2), eps=1e-6)
head.eval()

x_seq = torch.randn(1, 64, 384)  # [B, seq, dim]
t = torch.randn(1, 384)           # [B, dim] -- NOTE: t, NOT t_mod
# Source: diffsynth/models/wan_video_dit.py, line 406: self.head(x, t)

with torch.no_grad():
    head_out = head(x_seq, t)  # [1, 64, 64]
assert head_out.shape == torch.Size([1, 64, 16 * math.prod((1, 2, 2))]), (
    f"Expected [1, 64, 64], got {head_out.shape}")
print(f"Head input:  x_seq {x_seq.shape}, t {t.shape}")
print(f"Head output: {head_out.shape}  -- [B, seq_len, out_dim*prod(patch_size)]")

# Head + unpatchify end-to-end
x_video = rearrange(
    head_out, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
    f=4, h=4, w=4, x=1, y=2, z=2
)
assert x_video.shape == torch.Size([1, 16, 4, 8, 8]), (
    f"Expected [1, 16, 4, 8, 8], got {x_video.shape}")
print(f"Unpatchified: {x_video.shape}  -- [B, out_dim, F, H, W]")

# Head parameter counts
print(f"\nHead parameter counts:")
print(f"  modulation [1, 2, {384}]:  {head.modulation.numel():,} params (2-parameter: shift + scale)")
print(f"  norm (affine=False):       {sum(p.numel() for p in head.norm.parameters()):,} params")
head_linear_params = sum(p.numel() for p in head.head.parameters())
print(f"  head Linear({384}->{16 * math.prod((1,2,2))}): {head_linear_params:,} params")
total_head = sum(p.numel() for p in head.parameters())
print(f"  Head total:                {total_head:,} params")

# Verify modulation is 2-param, not 6-param
assert head.modulation.shape == torch.Size([1, 2, 384]), (
    f"Expected modulation [1, 2, 384], got {head.modulation.shape}")
print(f"\nKey: Head modulation is [1, 2, dim] (shift + scale only)")
print(f"     DiTBlock modulation is [1, 6, dim] (shift, scale, gate) x 2")

## Summary

### Key Takeaways
- **DiTBlock** wires five primitives (norm, self-attention, cross-attention, adaLN-Zero, FFN) into the core repeated unit of the diffusion backbone. The key asymmetry: self-attention and FFN use adaLN + gated residual, but cross-attention uses a simple additive residual with no gate and no modulation.
- **Patchify** uses `Conv3d(16, 384, kernel=(1,2,2), stride=(1,2,2))` followed by a flatten to convert video latents `[B, 16, F, H, W]` into token sequences `[B, seq_len, 384]`. The temporal kernel=1 preserves time resolution; spatial 2x2 patching reduces sequence length by 4x.
- **Head** applies 2-parameter adaLN (shift, scale) to project tokens back to output features, then unpatchify (einops rearrange) reconstructs the video shape. Head receives `t` [B, dim] from `time_embedding`, not `t_mod` [B, 6, dim] -- a critical distinction from DiTBlock's 6-parameter modulation.

### Source References
| Symbol | Location |
|--------|----------|
| GateModule | `diffsynth/models/wan_video_dit.py`, line 190 |
| DiTBlock.__init__ | `diffsynth/models/wan_video_dit.py`, line 197 |
| DiTBlock.forward | `diffsynth/models/wan_video_dit.py`, line 215 |
| Head | `diffsynth/models/wan_video_dit.py`, line 254 |
| patchify (Conv3d) | `diffsynth/models/wan_video_dit.py`, line 307, 340 |
| unpatchify | `diffsynth/models/wan_video_dit.py`, line 348 |

## Exercises

### Exercise 1 -- Temporal patching experiment
Change `patch_size` from `(1, 2, 2)` to `(2, 2, 2)` in the patchify demo cell. Recompute the Conv3d output shape and sequence length. How does this affect the temporal resolution? With the original `(1, 2, 2)`, the temporal dimension is preserved (F stays the same). With `(2, 2, 2)`, the temporal dimension is halved. Calculate the new sequence length for `(F=4, H=8, W=8)` input and verify your answer matches the code output.

### Exercise 2 -- Gate removal experiment
In the annotated forward trace (Stage 2), replace the gated residual `block.gate(x, gate_msa, sa_out)` with a simple residual `x + sa_out` (removing the gate). Run the cell and compare the output magnitude (using `.abs().mean()`) between the gated and ungated versions. Why is the gated version typically smaller at initialization? (Hint: the gate values are near-zero at init, so `gate * sa_out` is much smaller than `sa_out` alone.)

### Exercise 3 -- 30-layer parameter estimate
The production WanModel uses 30 DiTBlock layers with `dim=1536, num_heads=12, ffn_dim=8960`. Using the block parameter count formula from Cell 5 (total block params), estimate the total DiTBlock parameters for a 30-layer model. Then add the embedding and Head overhead (patch_embedding Conv3d, time_embedding MLP, time_projection, text_embedding MLP, Head). Compare your estimate to the published 1.3B parameter count.